In [ ]:
import random
import pandas as pd
import numpy as np
from present_cipher import PresentCipher
import time

def generate_cipher_data():
    """Generate 2^17 samples for each cipher variant"""
    print("Starting PRESENT cipher data generation...")

    # Fixed key for all variants
    fixed_key = 0x00112233445566778899

    # Create 3 cipher instances
    cipher_normal = PresentCipher(fixed_key, 'normal')
    cipher_modified = PresentCipher(fixed_key, 'modified')
    cipher_swapped = PresentCipher(fixed_key, 'swapped')

    # Generate 2^17 samples
    num_samples = 2**17  # 131,072

    data = []
    print(f"Generating {num_samples} samples for each cipher variant...")
    print("This will create a total of {0} data points (3 variants × {1} samples)".format(num_samples * 3, num_samples))

    start_time = time.time()

    for i in range(num_samples):
        if i % 5000 == 0:
            elapsed = time.time() - start_time
            print(f"Progress: {i:,}/{num_samples:,} ({i/num_samples*100:.1f}%) - Elapsed: {elapsed:.1f}s")

        # Generate random 64-bit plaintext
        plaintext = random.randint(0, 2**64 - 1)

        # Encrypt with all 3 variants
        cipher1 = cipher_normal.encrypt(plaintext)
        cipher2 = cipher_modified.encrypt(plaintext)
        cipher3 = cipher_swapped.encrypt(plaintext)

        # Add to dataset
        data.append({
            'plaintext': plaintext,
            'plaintext_hex': f"{plaintext:016X}",
            'ciphertext_normal': cipher1,
            'ciphertext_modified': cipher2,
            'ciphertext_swapped': cipher3,
            'ciphertext_normal_hex': f"{cipher1:016X}",
            'ciphertext_modified_hex': f"{cipher2:016X}",
            'ciphertext_swapped_hex': f"{cipher3:016X}"
        })

    total_time = time.time() - start_time
    print(f"Data generation completed in {total_time:.1f} seconds")
    print(f"Generated {len(data):,} unique plaintexts with 3 cipher variants each")

    return data

def save_to_excel(data):
    """Save data to Excel file with proper format"""
    print("Preparing data for Excel export...")

    excel_data = []

    for item in data:
        # Add 3 rows for each plaintext (one for each cipher variant)
        excel_data.append({
            'Plaintext_Hex': item['plaintext_hex'],
            'Plaintext_Int': item['plaintext'],
            'Ciphertext_Hex': item['ciphertext_normal_hex'],
            'Ciphertext_Int': item['ciphertext_normal'],
            'Cipher_Type': 'Normal',
            'Label': 0
        })
        excel_data.append({
            'Plaintext_Hex': item['plaintext_hex'],
            'Plaintext_Int': item['plaintext'],
            'Ciphertext_Hex': item['ciphertext_modified_hex'],
            'Ciphertext_Int': item['ciphertext_modified'],
            'Cipher_Type': 'Modified_Sbox',
            'Label': 1
        })
        excel_data.append({
            'Plaintext_Hex': item['plaintext_hex'],
            'Plaintext_Int': item['plaintext'],
            'Ciphertext_Hex': item['ciphertext_swapped_hex'],
            'Ciphertext_Int': item['ciphertext_swapped'],
            'Cipher_Type': 'Swapped_Sbox',
            'Label': 2
        })

    df = pd.DataFrame(excel_data)

    # Save to Excel
    excel_filename = 'cipher_data.xlsx'
    print(f"Saving {len(excel_data):,} rows to {excel_filename}...")
    df.to_excel(excel_filename, index=False)

    print(f"Excel file saved successfully!")
    print(f"File: {excel_filename}")
    print(f"Rows: {len(excel_data):,}")
    print(f"Columns: {list(df.columns)}")

    return df

def prepare_ml_data(data):
    """Prepare data for machine learning (CNN input format)"""
    print("Preparing data for machine learning...")

    X = []  # Features (ciphertext as 8 bytes)
    y = []  # Labels

    for item in data:
        # Process each cipher variant
        ciphertexts = [
            (item['ciphertext_normal'], 0),      # Normal cipher, label 0
            (item['ciphertext_modified'], 1),    # Modified cipher, label 1
            (item['ciphertext_swapped'], 2)      # Swapped cipher, label 2
        ]

        for ciphertext, label in ciphertexts:
            # Convert 64-bit ciphertext to 8 groups of 8-bits (8 bytes)
            feature_vector = []
            for i in range(8):
                byte_value = (ciphertext >> (8 * (7 - i))) & 0xFF  # Extract byte
                feature_vector.append(byte_value / 255.0)  # Normalize to [0,1]

            X.append(feature_vector)
            y.append(label)

    X = np.array(X)  # Shape: (393216, 8)
    y = np.array(y)  # Shape: (393216,)

    print(f"ML data prepared:")
    print(f"Features (X) shape: {X.shape}")
    print(f"Labels (y) shape: {y.shape}")
    print(f"Label distribution: {np.bincount(y)}")

    # Save ML data
    np.save('X_data.npy', X)
    np.save('y_data.npy', y)
    print("ML data saved as X_data.npy and y_data.npy")

    return X, y

def main():
    """Main function to generate all data"""
    print("=" * 60)
    print("PRESENT CIPHER DATA GENERATION")
    print("=" * 60)

    # Step 1: Generate cipher data
    data = generate_cipher_data()

    # Step 2: Save to Excel
    df = save_to_excel(data)

    # Step 3: Prepare ML data
    X, y = prepare_ml_data(data)

    print("\n" + "=" * 60)
    print("DATA GENERATION COMPLETE!")
    print("=" * 60)
    print("Files created:")
    print("1. cipher_data.xlsx - Excel file with all data")
    print("2. X_data.npy - Features for ML (ciphertext as 8 bytes)")
    print("3. y_data.npy - Labels for ML (0=Normal, 1=Modified, 2=Swapped)")
    print("\nNext steps:")
    print("1. Upload these files to Google Colab")
    print("2. Load the data in Colab using np.load()")
    print("3. Train the 1D CNN model")

    # Display sample data
    print("\nSample data preview:")
    print(df.head(10))

if __name__ == "__main__":
    # Set random seed for reproducibility
    random.seed(42)
    np.random.seed(42)

    main()


ModuleNotFoundError: No module named 'present_cipher'